# Previsão de volume e planejamento de produçãoPortfólio de modelos · Estudo 04. Nove métodos de série temporal competindo sob o mesmoprotocolo, e o erro do vencedor traduzido em estoque.**Semente fixa `20260907`.** As células estão na mesma ordem dos scripts que geraram orelatório, e o código foi recortado deles — não redigitado.**Sem `statsmodels`.** Holt-Winters e SARIMA estão implementados do zero, com numpy escipy. A seção 3 é a prova de que funcionam: parâmetros conhecidos são recuperados deprocessos simulados, e casos-limite dão a resposta analítica exata.Tempo total de execução: cerca de três minutos, quase todo no backtest.

---## 1. A sérieCinco anos semanais, cinco linhas de produto e quatro canais. Verão e inverno sãoespelhados de propósito, os eventos de e-commerce têm data móvel, e há duas quebras denível — a entrada num marketplace novo e uma mudança de comissão em outro.

In [ ]:
import numpy as npimport pandas as pdSEMENTE = 20260907rng = np.random.default_rng(SEMENTE)INICIO = pd.Timestamp("2021-09-06")   # segunda-feiraN_SEM = 261                           # 5 anos# ------------------------------------------------------------------ linhas# indice sazonal por MES (1,0 = media do ano). Interpolado para semanas depois.# Verao e inverno sao espelhados de proposito: e o que quebra um modelo unico.LINHAS = {    "Verao leve": dict(        base=9800, cresc=0.085, cv=0.11, devol=0.18,        mes=[1.42, 1.20, 0.95, 0.72, 0.55, 0.46, 0.48, 0.62, 0.95, 1.28, 1.45, 1.52]),    "Inverno": dict(        base=7400, cresc=0.062, cv=0.14, devol=0.16,        mes=[0.42, 0.46, 0.72, 1.18, 1.58, 1.72, 1.65, 1.34, 0.88, 0.58, 0.48, 0.44]),    "Basicos atemporais": dict(        base=14200, cresc=0.115, cv=0.08, devol=0.14,        mes=[0.94, 0.92, 1.00, 0.98, 1.02, 0.99, 1.00, 1.01, 1.00, 1.02, 1.08, 1.04]),    "Praia e fitness": dict(        base=5600, cresc=0.142, cv=0.19, devol=0.195,        mes=[1.68, 1.44, 1.02, 0.66, 0.44, 0.36, 0.38, 0.52, 0.86, 1.24, 1.60, 1.80]),    "Festa e ocasiao": dict(        base=3900, cresc=0.074, cv=0.24, devol=0.215,        mes=[0.72, 0.86, 0.92, 0.94, 1.24, 1.12, 0.88, 0.92, 0.96, 1.02, 1.18, 1.94]),}# ------------------------------------------------------------------ canais# share inicial e crescimento proprio: o mix muda ao longo dos 5 anosCANAIS = {    "Loja fisica":     dict(share0=0.30, cresc=-0.045, cv=0.07, sens_evento=0.25),    "Marketplace A":   dict(share0=0.42, cresc=0.055,  cv=0.10, sens_evento=1.00),    "Marketplace B":   dict(share0=0.00, cresc=0.240,  cv=0.16, sens_evento=1.35),    "Site proprio":    dict(share0=0.28, cresc=0.030,  cv=0.12, sens_evento=0.70),}ENTRADA_MKT_B = 70    # semana em que a marca entra no Marketplace BCORTE_MKT_A = 182     # semana da mudanca de comissao/algoritmo no Marketplace AQUEDA_MKT_A = 0.88    # nivel que sobra depois do cortedef domingo_n(ano, mes, n):    """N-esimo domingo do mes (Dia das Maes = 2o de maio, Pais = 2o de agosto)."""    d = pd.Timestamp(year=ano, month=mes, day=1)    d += pd.Timedelta(days=(6 - d.dayofweek) % 7)    return d + pd.Timedelta(weeks=n - 1)def sexta_black_friday(ano):    """Ultima sexta de novembro."""    d = pd.Timestamp(year=ano, month=11, day=30)    return d - pd.Timedelta(days=(d.dayofweek - 4) % 7)def calendario(semanas):    """Marca, para cada semana, quais eventos de varejo caem nela."""    ev = pd.DataFrame(index=semanas)    anos = sorted({d.year for d in semanas})    datas = {        "consumidor": [pd.Timestamp(a, 3, 15) for a in anos],        "maes": [domingo_n(a, 5, 2) for a in anos],        "namorados": [pd.Timestamp(a, 6, 12) for a in anos],        "pais": [domingo_n(a, 8, 2) for a in anos],        "black_friday": [sexta_black_friday(a) for a in anos],        "natal": [pd.Timestamp(a, 12, 25) for a in anos],    }    for nome, ds in datas.items():        col = np.zeros(len(semanas))        for d in ds:            # a semana comeca na segunda; o evento cai na semana que o contem.            # Sem a checagem dos 7 dias, um evento POSTERIOR ao fim da serie            # (Black Friday e Natal de 2026) seria marcado na ultima semana.            i = np.searchsorted(semanas, d, side="right") - 1            if 0 <= i < len(semanas) and (d - semanas[i]).days < 7:                col[i] = 1        ev[nome] = col    # a semana ANTERIOR a Black Friday ja tem antecipacao    ev["pre_black"] = np.r_[ev.black_friday.to_numpy()[1:], 0]    # liquidacao de virada de estacao: fim de fevereiro e fim de agosto    ev["virada"] = [(1 if (d.month == 2 and d.day >= 20)                     or (d.month == 8 and d.day >= 20) else 0) for d in semanas]    return ev# multiplicador de cada evento, por linha de produtoEFEITO = {    #                    consum  maes  namor  pais  BF    pre   natal virada    "Verao leve":        [1.18, 1.22, 1.14, 1.06, 2.35, 1.28, 1.42, 1.34],    "Inverno":           [1.12, 1.34, 1.26, 1.22, 2.62, 1.30, 1.20, 1.46],    "Basicos atemporais":[1.15, 1.16, 1.10, 1.12, 2.10, 1.22, 1.28, 1.22],    "Praia e fitness":   [1.22, 1.10, 1.08, 1.04, 2.18, 1.24, 1.38, 1.30],    "Festa e ocasiao":   [1.10, 1.48, 1.86, 1.08, 1.92, 1.18, 2.05, 1.12],}COLS_EV = ["consumidor", "maes", "namorados", "pais",           "black_friday", "pre_black", "natal", "virada"]def perfil_semanal(mes_idx):    """Interpola os 12 indices mensais para 52 semanas, de forma periodica."""    x = np.arange(-1, 14)                      # meses com borda para ciclar    y = np.r_[mes_idx[-1], mes_idx, mes_idx[0], mes_idx[1]]    # posicao de cada semana no eixo de meses (0 = meio de janeiro)    pos = np.arange(52) * 12 / 52    return np.interp(pos, x + 0.5, y)def gerar():    semanas = pd.date_range(INICIO, periods=N_SEM, freq="W-MON")    ev = calendario(semanas)    t = np.arange(N_SEM)    sem_ano = np.array([min(d.isocalendar().week, 52) - 1 for d in semanas])    linhas = []    for nome, p in LINHAS.items():        perfil = perfil_semanal(np.array(p["mes"]))        saz = perfil[sem_ano]        tend = p["base"] * (1 + p["cresc"]) ** (t / 52)        mult_ev = np.ones(N_SEM)        for col, m in zip(COLS_EV, EFEITO[nome]):            mult_ev *= np.where(ev[col].to_numpy() > 0, m, 1.0)        # promocao propria: pulsos raros, independentes do calendario        promo = np.where(rng.random(N_SEM) < 0.055,                         rng.uniform(1.18, 1.55, N_SEM), 1.0)        mu = tend * saz * mult_ev * promo        bruto = rng.gamma(1 / p["cv"] ** 2, mu * p["cv"] ** 2)        # devolucao: varia em torno da taxa da linha, e sobe em semana de pico        pico = mult_ev / mult_ev.mean()        taxa = np.clip(p["devol"] * (0.85 + 0.30 * pico)                       + rng.normal(0, 0.012, N_SEM), 0.05, 0.42)        linhas.append(pd.DataFrame({            "semana": semanas, "linha": nome,            "pecas_brutas": np.round(bruto).astype(int),            "taxa_devolucao": taxa.round(4),            "promo": (promo > 1).astype(int),        }))    d = pd.concat(linhas, ignore_index=True)    d["pecas_liquidas"] = np.round(d.pecas_brutas * (1 - d.taxa_devolucao)).astype(int)    # ---------------------------------------------------------------- canais    # o mix de canal e uma repartição do volume total da semana    tot = d.groupby("semana").pecas_liquidas.sum()    pesos = {}    for c, q in CANAIS.items():        w = q["share0"] * (1 + q["cresc"]) ** (t / 52)        if c == "Marketplace B":            # canal novo: entra do zero e cresce rapido (curva logistica)            w = 0.34 / (1 + np.exp(-(t - ENTRADA_MKT_B - 26) / 13))            w[:ENTRADA_MKT_B] = 0.0        if c == "Marketplace A":            w = np.where(t >= CORTE_MKT_A, w * QUEDA_MKT_A, w)        pesos[c] = w    W = pd.DataFrame(pesos, index=semanas)    W = W.div(W.sum(axis=1), axis=0)    can = []    for c in CANAIS:        base = tot.to_numpy() * W[c].to_numpy()        # cada canal reage aos eventos com intensidade propria        s = CANAIS[c]["sens_evento"]        m = np.ones(N_SEM)        for col, ef in zip(COLS_EV, EFEITO["Basicos atemporais"]):            m *= np.where(ev[col].to_numpy() > 0, 1 + (ef - 1) * s, 1.0)        m /= m.mean()        v = rng.gamma(1 / CANAIS[c]["cv"] ** 2,                      base * m * CANAIS[c]["cv"] ** 2)        can.append(pd.DataFrame({"semana": semanas, "canal": c,                                 "pecas_liquidas": np.round(v).astype(int)}))    dc = pd.concat(can, ignore_index=True)    return d, dc, ev.reset_index(names="semana")

In [ ]:
d, dc, ev = gerar()d.to_csv("vendas_linha.csv", index=False)dc.to_csv("vendas_canal.csv", index=False)ev.to_csv("calendario.csv", index=False)def n(x, c=0):    return f"{x:,.{c}f}".replace(",", "X").replace(".", ",").replace("X", ".")print(f"semanas: {d.semana.nunique()}   "      f"de {d.semana.min():%d/%m/%Y} a {d.semana.max():%d/%m/%Y}")print(f"linhas: {d.linha.nunique()}   canais: {dc.canal.nunique()}")print()r = d.groupby("linha").agg(    pecas_ano=("pecas_liquidas", lambda s: s.sum() / 5),    devol=("taxa_devolucao", "mean"),    cv=("pecas_liquidas", lambda s: s.std() / s.mean()))r["share"] = r.pecas_ano / r.pecas_ano.sum()print("por linha de produto (media anual):")for k, v in r.iterrows():    print(f"  {k:20s} {n(v.pecas_ano):>10s} pecas/ano   share {v.share:5.1%}"          f"   devolucao {v.devol:5.1%}   CV {v.cv:.2f}")print()print("crescimento de volume, ano a ano (total liquido):")a = d.assign(ano=d.semana.dt.year).groupby("ano").pecas_liquidas.sum()a = a[(a.index > 2021) & (a.index < 2026)]for i in range(1, len(a)):    print(f"  {a.index[i]}: {n(a.iloc[i]):>9s}   "          f"{(a.iloc[i]/a.iloc[i-1]-1):+6.1%}")print()print("mix de canal (share do volume):")m = dc.assign(ano=dc.semana.dt.year).pivot_table(    index="ano", columns="canal", values="pecas_liquidas", aggfunc="sum")print((m.div(m.sum(axis=1), axis=0) * 100).round(1).to_string())print()print("semanas de evento encontradas:")print("  " + "  ".join(f"{c}={int(ev[c].sum())}" for c in COLS_EV))

---## 2. Os nove métodosTodos expõem a mesma interface `prever(y, h) -> array de tamanho h`, e nenhum enxerga ofuturo. Só a regressão harmônica e o gradient boosting recebem o calendário — e éexatamente essa a pergunta do estudo.

In [ ]:
import numpy as npfrom scipy.optimize import minimizefrom sklearn.ensemble import HistGradientBoostingRegressorM = 52   # periodo sazonal: semanas no ano# ==================================================================== baselinesdef naive(y, h, **kw):    return np.repeat(y[-1], h)def naive_sazonal(y, h, m=M, **kw):    """Repete o mesmo periodo do ano anterior. E a referencia a bater."""    base = y[-m:]    return np.array([base[i % m] for i in range(h)])def media_movel(y, h, janela=8, **kw):    return np.repeat(y[-janela:].mean(), h)def deriva_sazonal(y, h, m=M, **kw):    """Naive sazonal mais a tendencia media entre os dois ultimos anos."""    if len(y) < 2 * m:        return naive_sazonal(y, h, m)    passo = (y[-m:].mean() - y[-2 * m:-m].mean()) / m    base = naive_sazonal(y, h, m)    return base + passo * (np.arange(h) + 1)# ==================================================================== Holt-Wintersdef _hw_init(y, m, tipo):    """Inicializacao por decomposicao classica.    Tirar os indices sazonais so do PRIMEIRO ciclo os deixa ruidosos, e a    recursao compensa isso deixando a sazonalidade absorver a tendencia -    o modelo fica otimo dentro da amostra e ruim na previsao. Estimar os    indices sobre TODOS os ciclos, depois de remover uma reta, resolve.    """    n = len(y)    t = np.arange(n)    A = np.column_stack([np.ones(n), t])    coef, *_ = np.linalg.lstsq(A, y, rcond=None)    tend = A @ coef    resid = y / np.maximum(tend, 1e-9) if tipo == "mult" else y - tend    pos = t % m    s0 = np.array([resid[pos == k].mean() if (pos == k).any() else                   (1.0 if tipo == "mult" else 0.0) for k in range(m)])    # normaliza: media 1 no multiplicativo, media 0 no aditivo    s0 = s0 / s0.mean() if tipo == "mult" else s0 - s0.mean()    return float(coef[0]), float(coef[1]), s0def _hw_ajuste(y, par, m, tipo):    """Recursao de Holt-Winters. Devolve (nivel, tendencia, sazonal, erros)."""    a, b, g = par    n = len(y)    l0, b0, s0 = _hw_init(y, m, tipo)    L, B = l0, b0    S = list(s0)    err = np.zeros(n)    for t in range(n):        st = S[t]        if tipo == "mult":            f = (L + B) * st            err[t] = y[t] - f            Ln = a * (y[t] / max(st, 1e-6)) + (1 - a) * (L + B)            Bn = b * (Ln - L) + (1 - b) * B            Sn = g * (y[t] / max(Ln, 1e-6)) + (1 - g) * st        else:            f = L + B + st            err[t] = y[t] - f            Ln = a * (y[t] - st) + (1 - a) * (L + B)            Bn = b * (Ln - L) + (1 - b) * B            Sn = g * (y[t] - Ln) + (1 - g) * st        L, B = Ln, Bn        S.append(Sn)    return L, B, np.array(S), err# Limites do espaco de busca. Sem eles o otimizador leva gamma para perto de 1# e beta para zero: o ajuste dentro da amostra melhora e a previsao piora.LIM_HW = [(0.02, 0.65), (0.001, 0.20), (0.01, 0.45)]def _hw_sse(par, y, m, tipo):    if any(v < lo or v > hi for v, (lo, hi) in zip(par, LIM_HW)):        return 1e18    try:        _, _, _, e = _hw_ajuste(y, par, m, tipo)    except Exception:        return 1e18    if not np.all(np.isfinite(e)):        return 1e18    return float(e @ e)def holt_winters(y, h, m=M, tipo="add", amortecido=False, **kw):    """ETS(A,A,A) ou ETS(A,A,M) com parametros por minimos quadrados."""    if len(y) < 2 * m:        return naive_sazonal(y, h, m)    melhor, sse_min = None, np.inf    for p0 in [(0.20, 0.05, 0.15), (0.45, 0.02, 0.30), (0.06, 0.01, 0.05)]:        r = minimize(_hw_sse, p0, args=(y, m, tipo), method="L-BFGS-B",                     bounds=LIM_HW)        if r.fun < sse_min:            sse_min, melhor = r.fun, r.x    a, b, g = [np.clip(v, lo, hi) for v, (lo, hi) in zip(melhor, LIM_HW)]    L, B, S, _ = _hw_ajuste(y, (a, b, g), m, tipo)    n = len(y)    phi = 0.92 if amortecido else 1.0    saz = S[n:n + m] if len(S) >= n + m else S[-m:]    out = np.empty(h)    for i in range(h):        passo = sum(phi ** (k + 1) for k in range(i + 1))        st = saz[i % m]        out[i] = (L + passo * B) * st if tipo == "mult" else (L + passo * B) + st    return np.maximum(out, 0)def holt_winters_mult(y, h, **kw):    return holt_winters(y, h, tipo="mult", **kw)# ==================================================================== SARIMAdef _poly_dif(d, D, s):    """Coeficientes de (1-B)^d (1-B^s)^D, com p[0] = 1."""    p = np.array([1.0])    for _ in range(d):        p = np.convolve(p, [1.0, -1.0])    for _ in range(D):        q = np.zeros(s + 1)        q[0], q[s] = 1.0, -1.0        p = np.convolve(p, q)    return pdef _aplicar_dif(y, d, D, s):    p = _poly_dif(d, D, s)    k = len(p) - 1    if len(y) <= k:        return np.array([])    return np.array([p @ y[t::-1][:len(p)] for t in range(k, len(y))])def _expandir(coef, ordem, s_ordem, s, sinal):    """Multiplica o polinomio nao sazonal pelo sazonal.    sinal = -1 para AR (1 - phi B), +1 para MA (1 + theta B).    """    p = np.r_[1.0, sinal * np.asarray(coef[:ordem], dtype=float)]    q = np.array([1.0])    if s_ordem:        q = np.zeros(s * s_ordem + 1)        q[0] = 1.0        for i in range(s_ordem):            q[s * (i + 1)] = sinal * coef[ordem + i]    return np.convolve(p, q)def _sarima_res(par, w, p, q, P, Q, s):    """Residuos por soma de quadrados condicional (CSS) na serie diferenciada.    A parte AR e uma convolucao, entao sai vetorizada de uma vez. So a parte    MA precisa de laco, porque realimenta o proprio residuo. Com defasagem    sazonal 52 essa diferenca vale dez vezes em tempo de execucao.    """    ar = _expandir(par, p, P, s, -1.0)    ma = _expandir(par[p + P:], q, Q, s, +1.0)    n = len(w)    k = max(len(ar), len(ma)) - 1    ar_part = np.convolve(w, ar)[:n]          # sum_i ar[i] * w[t-i]    if len(ma) == 1:                          # modelo puramente AR: sem laco        e = ar_part.copy()        e[:k] = 0.0        return e, k    ma_rev = ma[1:][::-1]                     # para casar com a janela e[t-L:t]    L = len(ma_rev)    e = np.zeros(n)    for t in range(k, n):        e[t] = ar_part[t] - ma_rev @ e[t - L:t]    return e, kdef _sarima_sse(par, w, p, q, P, Q, s):    if np.max(np.abs(par)) > 0.995:        return 1e18    e, k = _sarima_res(par, w, p, q, P, Q, s)    if not np.all(np.isfinite(e)):        return 1e18    return float(e[k:] @ e[k:])def sarima_ajustar(y, ordem, s=M):    """Estima por CSS. Devolve dict com parametros, AICc e residuos."""    (p, d, q), (P, D, Q) = ordem    w = _aplicar_dif(y, d, D, s)    npar = p + P + q + Q    if len(w) < 3 * (npar + 1) or len(w) < 20:        return None    # Constante so quando d + D <= 1. Com duas diferenciacoes ela implicaria    # tendencia quadratica na serie original - quase nunca o que se quer.    mu = float(w.mean()) if (d + D) <= 1 else 0.0    w = w - mu    if npar == 0:        e, k = np.zeros(len(w)), 0        sse = float(w @ w)    else:        melhor, sse = None, np.inf        for p0 in ([np.full(npar, 0.1), np.full(npar, -0.1)]):            r = minimize(_sarima_sse, p0, args=(w, p, q, P, Q, s),                         method="Nelder-Mead",                         options=dict(maxiter=1400, xatol=1e-4, fatol=1e-4))            if r.fun < sse:                sse, melhor = r.fun, r.x        e, k = _sarima_res(melhor, w, p, q, P, Q, s)    n_ef = len(w) - (0 if npar == 0 else k)    sigma2 = max(sse / n_ef, 1e-12)    aic = n_ef * np.log(sigma2) + 2 * (npar + 1)    aicc = aic + (2 * (npar + 1) * (npar + 2)) / max(n_ef - npar - 2, 1)    return dict(par=(melhor if npar else np.array([])), ordem=ordem, s=s,                aicc=aicc, sigma=np.sqrt(sigma2), e=e, w=w, y=y, mu=mu)def sarima_prever(fit, h):    (p, d, q), (P, D, Q) = fit["ordem"]    s = fit["s"]    par, w, e, y, mu = fit["par"], fit["w"], fit["e"], fit["y"], fit["mu"]    ar = _expandir(par, p, P, s, -1.0) if len(par) else np.array([1.0])    ma = _expandir(par[p + P:], q, Q, s, +1.0) if len(par) else np.array([1.0])    # projeta a serie diferenciada    W = list(w)    E = list(e)    for _ in range(h):        v = 0.0        for i in range(1, len(ar)):            v -= ar[i] * W[-i]        for j in range(1, len(ma)):            v += ma[j] * E[-j]        W.append(v)        E.append(0.0)    w_fut = np.array(W[len(w):]) + mu    # integra de volta: w[t] = sum_k pol[k] y[t-k]  =>  y[t] = w[t] - sum_{k>=1} pol[k] y[t-k]    pol = _poly_dif(d, D, s)    Y = list(y)    for i in range(h):        v = w_fut[i]        for k in range(1, len(pol)):            v -= pol[k] * Y[-k]        Y.append(v)    return np.maximum(np.array(Y[len(y):]), 0)GRADE_SARIMA = [((p, 1, q), (P, 1, Q))                for p in (0, 1, 2) for q in (0, 1, 2)                for P in (0, 1) for Q in (0, 1)]def sarima_escolher_ordem(y, s=M, grade=None):    """Seleciona a ordem uma vez, por AICc. Depois so os parametros sao reajustados."""    melhor, best = None, np.inf    for o in (grade or GRADE_SARIMA):        f = sarima_ajustar(y, o, s)        if f and f["aicc"] < best:            best, melhor = f["aicc"], o    return melhordef sarima(y, h, ordem=None, s=M, **kw):    o = ordem or ((1, 1, 1), (0, 1, 1))    f = sarima_ajustar(y, o, s)    if f is None:        return naive_sazonal(y, h, s)    return sarima_prever(f, h)# ==================================================================== regressao harmonicadef _matriz_harmonica(t, K, m=M):    X = [np.ones_like(t, dtype=float), t / m]    for k in range(1, K + 1):        X.append(np.sin(2 * np.pi * k * t / m))        X.append(np.cos(2 * np.pi * k * t / m))    return np.column_stack(X)def regressao_harmonica(y, h, K=6, eventos=None, ev_fut=None, log=True, **kw):    """Tendencia + K harmonicas de Fourier + dummies de evento, por OLS.    E o unico metodo da comparacao que enxerga o calendario. Todos os outros    so veem a serie, e por isso nao tem como saber que a Black Friday muda de    semana a cada ano.    """    n = len(y)    t = np.arange(n)    z = np.log(np.maximum(y, 1e-6)) if log else y.astype(float)    X = _matriz_harmonica(t, K)    Xf = _matriz_harmonica(np.arange(n, n + h), K)    if eventos is not None:        X = np.column_stack([X, eventos])        Xf = np.column_stack([Xf, ev_fut])    beta, *_ = np.linalg.lstsq(X, z, rcond=None)    pred = Xf @ beta    if log:        resid = z - X @ beta        # correcao de retransformacao (Duan): a media do log nao e o log da media        pred = np.exp(pred) * np.mean(np.exp(resid))    return np.maximum(pred, 0)# ==================================================================== gradient boostingdef _features_gbm(y, t, eventos, lags=(1, 2, 3, 4, 8, 52, 104)):    n = len(y)    cols, nomes = [], []    for L in lags:        v = np.full(n, np.nan)        v[L:] = y[:-L]        cols.append(v)        nomes.append(f"lag{L}")    for j in (4, 13, 52):        v = np.full(n, np.nan)        for i in range(j, n):            v[i] = y[i - j:i].mean()        cols.append(v)        nomes.append(f"mm{j}")    sem = t % M    cols += [np.sin(2 * np.pi * sem / M), np.cos(2 * np.pi * sem / M),             np.sin(4 * np.pi * sem / M), np.cos(4 * np.pi * sem / M), t / M]    nomes += ["sin1", "cos1", "sin2", "cos2", "tend"]    X = np.column_stack(cols + [eventos]) if eventos is not None \        else np.column_stack(cols)    return X, nomesdef gbm(y, h, eventos=None, ev_fut=None, semente=7, **kw):    """Recursivo: prevê um passo, realimenta, prevê o seguinte."""    n = len(y)    t = np.arange(n)    X, _ = _features_gbm(y, t, eventos)    ok = ~np.isnan(X).any(axis=1)    if ok.sum() < 40:        return naive_sazonal(y, h)    mod = HistGradientBoostingRegressor(        max_iter=260, learning_rate=0.06, max_depth=4, min_samples_leaf=8,        l2_regularization=1.0, random_state=semente)    mod.fit(X[ok], y[ok])    hist = list(y)    ev_all = None    if eventos is not None:        ev_all = np.vstack([eventos, ev_fut])    for i in range(h):        ya = np.array(hist)        ta = np.arange(len(ya))        e = ev_all[:len(ya)] if ev_all is not None else None        Xa, _ = _features_gbm(ya, ta, e)        linha = Xa[-1:].copy()        if np.isnan(linha).any():            linha = np.nan_to_num(linha, nan=np.nanmedian(Xa))        hist.append(float(mod.predict(linha)[0]))    return np.maximum(np.array(hist[n:]), 0)# ==================================================================== registroMETODOS = {    "Naive": naive,    "Naive sazonal": naive_sazonal,    "Média móvel 8": media_movel,    "Deriva sazonal": deriva_sazonal,    "Holt-Winters aditivo": holt_winters,    "Holt-Winters multiplicativo": holt_winters_mult,    "SARIMA": sarima,    "Regressão harmônica": regressao_harmonica,    "Gradient boosting": gbm,}# quais metodos recebem o calendario de eventosUSA_CALENDARIO = {"Regressão harmônica", "Gradient boosting"}FAMILIA = {    "Naive": "Baselines", "Naive sazonal": "Baselines",    "Média móvel 8": "Baselines", "Deriva sazonal": "Baselines",    "Holt-Winters aditivo": "Suavização exponencial",    "Holt-Winters multiplicativo": "Suavização exponencial",    "SARIMA": "SARIMA",    "Regressão harmônica": "Regressão com calendário",    "Gradient boosting": "Machine learning",}

---## 3. A prova de que as implementações estão certasCódigo próprio não vale nada sem prova. Catorze testes: recuperação de parâmetros emprocessos simulados, casos-limite com resposta analítica conhecida, e uma verificação de**consistência** — o viés dos termos sazonais do SARIMA precisa encolher quando aamostra cresce, e encolhe.Se algum teste falhar, esta célula levanta erro e o resto do notebook não deve ser lido.

In [ ]:
import numpy as nprng = np.random.default_rng(4242)OK, FALHA = [], []def checar(nome, cond, detalhe=""):    (OK if cond else FALHA).append(nome)    print(f"  [{'ok  ' if cond else 'FALHA'}] {nome}{'  ' + detalhe if detalhe else ''}")# ------------------------------------------------------------------ 1. algebraprint("1. Diferenciacao e integracao sao operacoes inversas")s = 12y = np.cumsum(rng.normal(0, 1, 300)) + 10 * np.sin(2 * np.pi * np.arange(300) / s) + 100w = _aplicar_dif(y, 1, 1, s)pol = _poly_dif(1, 1, s)k = len(pol) - 1rec = list(y[:k])for i in range(len(w)):    v = w[i] - sum(pol[j] * rec[-j] for j in range(1, len(pol)))    rec.append(v)checar("integrar(diferenciar(y)) == y", np.allclose(rec, y, atol=1e-8),       f"erro max {np.max(np.abs(np.array(rec) - y)):.2e}")print("\n2. Expansao do polinomio multiplicativo")# (1 - 0.5B)(1 - 0.3B^4) = 1 - 0.5B - 0.3B^4 + 0.15B^5e = _expandir(np.array([0.5, 0.3]), 1, 1, 4, -1.0)esperado = np.array([1, -0.5, 0, 0, -0.3, 0.15])checar("(1-0.5B)(1-0.3B^4) expandido", np.allclose(e, esperado, atol=1e-12),       str(np.round(e, 3)))# ------------------------------------------------------------------ 3. SARIMAdef simular_sarima(n, phi, theta, Phi, Theta, s, sigma=1.0, d=1, D=1, semente=1):    """Gera uma serie cujo SARIMA(1,d,1)(1,D,1)_s tem parametros conhecidos."""    r = np.random.default_rng(semente)    burn = 400    N = n + burn    ar = _expandir(np.array([phi, Phi]), 1, 1, s, -1.0)    ma = _expandir(np.array([theta, Theta]), 1, 1, s, +1.0)    e = r.normal(0, sigma, N)    w = np.zeros(N)    for t in range(len(ar) + len(ma), N):        v = e[t]        for j in range(1, len(ma)):            v += ma[j] * e[t - j]        for i in range(1, len(ar)):            v -= ar[i] * w[t - i]        w[t] = v    # integra para obter a serie no nivel    pol = _poly_dif(d, D, s)    y = list(np.zeros(len(pol) - 1))    for i in range(len(pol) - 1, N):        v = w[i] - sum(pol[j] * y[-j] for j in range(1, len(pol)))        y.append(v)    return np.array(y[burn:]) + 500.0print("\n3. SARIMA: os parametros nao sazonais sao recuperados;")print("   os sazonais tem vies de amostra finita, que precisa ENCOLHER com n")VERD = dict(phi=0.55, theta=-0.35, Phi=0.30, Theta=-0.45)ALVO = [VERD["phi"], VERD["Phi"], VERD["theta"], VERD["Theta"]]   # ar, sar, ma, smaest_por_n = {}for n in (600, 1800):    y = simular_sarima(n, s=12, semente=3, **VERD)    est_por_n[n] = sarima_ajustar(y, ((1, 1, 1), (1, 1, 1)), s=12)["par"]e6, e18 = est_por_n[600], est_por_n[1800]checar(f"   phi   n=1800: {e18[0]:+.3f} vs {ALVO[0]:+.3f}", abs(e18[0] - ALVO[0]) < 0.10)checar(f"   theta n=1800: {e18[2]:+.3f} vs {ALVO[2]:+.3f}", abs(e18[2] - ALVO[2]) < 0.10)# Estimacao por CSS condiciona os residuos iniciais a zero. Com defasagem# sazonal isso custa caro em amostra pequena: o vies e real, mas some com n.for k, nome in [(1, "Phi"), (3, "Theta")]:    v6, v18 = abs(e6[k] - ALVO[k]), abs(e18[k] - ALVO[k])    checar(f"   {nome}: sinal correto e vies cai de {v6:.3f} para {v18:.3f}",           np.sign(e18[k]) == np.sign(ALVO[k]) and v18 < v6 * 0.80,           f"({e6[k]:+.3f} -> {e18[k]:+.3f})")# o que importa no fim nao e o parametro, e a previsaoy = simular_sarima(700, s=12, semente=9, **VERD)tr, te = y[:-24], y[-24:]f = sarima_prever(sarima_ajustar(tr, ((1, 1, 1), (1, 1, 1)), s=12), 24)checar("   e a previsao bate o naive sazonal no proprio processo",       np.mean(np.abs(f - te)) < np.mean(np.abs(naive_sazonal(tr, 24, m=12) - te)),       f"EAM {np.mean(np.abs(f - te)):.2f} contra "       f"{np.mean(np.abs(naive_sazonal(tr, 24, m=12) - te)):.2f}")print("\n4. Casos-limite com resposta analitica")z = np.arange(1, 121) * 1.0 + 50f = sarima_prever(sarima_ajustar(z, ((0, 1, 0), (0, 0, 0)), s=12), 5)checar("ARIMA(0,1,0) com deriva numa reta -> continua a reta",       np.allclose(f, [171, 172, 173, 174, 175], atol=1e-6), str(np.round(f, 2)))base = 100 + 20 * np.sin(2 * np.pi * np.arange(120) / 12)f = sarima_prever(sarima_ajustar(base, ((0, 0, 0), (0, 1, 0)), s=12), 12)checar("SARIMA(0,0,0)(0,1,0)_12 -> repete o ciclo anterior",       np.allclose(f, base[-12:], atol=1e-6),       f"erro max {np.max(np.abs(f - base[-12:])):.2e}")# ------------------------------------------------------------------ 5. Holt-Wintersprint("\n5. Holt-Winters num sinal deterministico")n, m = 260, 52t = np.arange(n + 13)sinal = (1000 + 3.0 * t) * (1 + 0.35 * np.sin(2 * np.pi * t / m))yh, futuro = sinal[:n], sinal[n:]pm = holt_winters(yh, 13, m=m, tipo="mult")erro = np.mean(np.abs(pm - futuro) / futuro)checar("HW multiplicativo segue tendencia + sazonalidade (EPAM < 3%)",       erro < 0.03, f"EPAM {erro:.2%}")ruido = sinal + rng.normal(0, 25, len(sinal))pa = holt_winters(ruido[:n], 13, m=m, tipo="add")pn = naive_sazonal(ruido[:n], 13, m=m)e_hw = np.mean(np.abs(pa - futuro))e_ns = np.mean(np.abs(pn - futuro))checar("HW aditivo bate o naive sazonal em serie com tendencia",       e_hw < e_ns, f"EAM {e_hw:.1f} contra {e_ns:.1f}")# ------------------------------------------------------------------ 6. harmonicaprint("\n6. Regressao harmonica recupera uma senoide conhecida")t = np.arange(200)verdade = np.exp(6.0 + 0.002 * t + 0.30 * np.sin(2 * np.pi * t / 52)                 + 0.15 * np.cos(4 * np.pi * t / 52))p = regressao_harmonica(verdade[:180], 20, K=3)erro = np.mean(np.abs(p - verdade[180:]) / verdade[180:])checar("EPAM < 1% em sinal sem ruido", erro < 0.01, f"EPAM {erro:.3%}")X = np.zeros((200, 1))X[[30, 82, 134, 186], 0] = 1          # um evento a cada 52 semanascom_evento = verdade * (1 + 0.8 * X[:, 0])p = regressao_harmonica(com_evento[:180], 20, K=3,                        eventos=X[:180], ev_fut=X[180:])i = 186 - 180checar("captura o pico de evento na semana certa",       abs(p[i] / com_evento[186] - 1) < 0.10,       f"previsto {p[i]:.0f} vs real {com_evento[186]:.0f}")sem = regressao_harmonica(com_evento[:180], 20, K=3)checar("sem o calendario, o mesmo modelo erra o pico",       abs(sem[i] / com_evento[186] - 1) > 0.30,       f"previsto {sem[i]:.0f} vs real {com_evento[186]:.0f}")print("\n" + "=" * 62)print(f"{len(OK)} testes passaram, {len(FALHA)} falharam")if FALHA:    print("FALHAS:", FALHA)    raise SystemExit(1)print("As implementacoes proprias estao validadas.")

---## 4. O backtest de origem móvelDoze origens por série, treze semanas de teste em cada. A ordem do SARIMA é escolhidauma única vez, na primeira janela — reescolher a cada origem seria usar o teste paracalibrar o modelo.Esta é a célula lenta: cerca de setenta segundos.

In [ ]:
import numpy as npimport pandas as pdfrom scipy import statsH = 13              # horizonte: 13 semanas cobrem o lead time de confeccaoTREINO_MIN = 156    # 3 anos: o minimo para ver tres ciclos anuaisPASSO = 8           # espacamento entre origensCOLS_EV = ["consumidor", "maes", "namorados", "pais",           "black_friday", "pre_black", "natal", "virada"]def origens(n, h=H, treino_min=TREINO_MIN, passo=PASSO):    return list(range(treino_min, n - h + 1, passo))def mase_denominador(y_tr, m=M):    """EAM do naive sazonal DENTRO da amostra de treino - a escala do MASE."""    if len(y_tr) <= m:        return np.mean(np.abs(np.diff(y_tr)))    return np.mean(np.abs(y_tr[m:] - y_tr[:-m]))def rodar(series, ev, h=H, metodos=None, verbose=True):    """series: dict nome -> array. ev: DataFrame de eventos alinhado as semanas."""    metodos = metodos or METODOS    EV = ev[COLS_EV].to_numpy(dtype=float)    linhas, ordens = [], {}    for nome, y in series.items():        y = np.asarray(y, dtype=float)        ors = origens(len(y), h)        # a ordem do SARIMA e escolhida UMA vez, na primeira janela de treino.        # Reescolher a cada origem seria usar o backtest para calibrar o modelo.        ordens[nome] = sarima_escolher_ordem(y[:ors[0]], s=M, grade=GRADE_SARIMA)        if verbose:            print(f"  {nome:22s} ordem SARIMA {ordens[nome]}  "                  f"{len(ors)} origens", flush=True)        for o in ors:            y_tr, y_te = y[:o], y[o:o + h]            den = mase_denominador(y_tr)            for mnome, fn in metodos.items():                kw = {}                if mnome in USA_CALENDARIO:                    kw = dict(eventos=EV[:o], ev_fut=EV[o:o + h])                if mnome == "SARIMA":                    kw = dict(ordem=ordens[nome])                try:                    p = np.asarray(fn(y_tr, h, **kw), dtype=float)                except Exception:                    p = naive_sazonal(y_tr, h)                if p.shape != (h,) or not np.all(np.isfinite(p)):                    p = naive_sazonal(y_tr, h)                for i in range(h):                    linhas.append((nome, mnome, o, i + 1, y_te[i], p[i], den))    return pd.DataFrame(linhas, columns=["serie", "metodo", "origem",                                         "horizonte", "real", "previsto",                                         "den_mase"]), ordensdef metricas(bt, por=("serie", "metodo")):    d = bt.copy()    d["erro"] = d.previsto - d.real    d["abs"] = d.erro.abs()    g = d.groupby(list(por))    r = g.apply(lambda x: pd.Series({        "EAM": x["abs"].mean(),        "REQM": np.sqrt((x.erro ** 2).mean()),        "MASE": (x["abs"] / x.den_mase).mean(),        "EPAM": (x["abs"] / x.real.clip(lower=1)).mean(),        "sEPAM": (2 * x["abs"] / (x.real.abs() + x.previsto.abs()).clip(lower=1)).mean(),        "vies": x.erro.mean() / x.real.mean(),        "n": len(x),    }), include_groups=False)    return r.reset_index()def diebold_mariano(bt, serie, m1, m2, h=H, potencia=1):    """Teste de Diebold-Mariano com correcao de Harvey-Leybourne-Newbold.    A pergunta nao e "qual EAM foi menor", e "a diferenca sobrevive ao acaso".    Como previsoes de h passos se sobrepoem, a variancia precisa de correcao    HAC; e como a amostra e pequena, entra a correcao HLN.    """    a = bt[(bt.serie == serie) & (bt.metodo == m1)].sort_values(["origem", "horizonte"])    b = bt[(bt.serie == serie) & (bt.metodo == m2)].sort_values(["origem", "horizonte"])    L1 = np.abs(a.previsto.to_numpy() - a.real.to_numpy()) ** potencia    L2 = np.abs(b.previsto.to_numpy() - b.real.to_numpy()) ** potencia    d = L1 - L2    n = len(d)    dbar = d.mean()    dc = d - dbar    gama0 = (dc @ dc) / n    var = gama0    for k in range(1, h):        gk = (dc[k:] @ dc[:-k]) / n        var += 2 * (1 - k / h) * gk        # janela de Bartlett    var = max(var, 1e-18)    dm = dbar / np.sqrt(var / n)    corr = np.sqrt(max((n + 1 - 2 * h + h * (h - 1) / n) / n, 1e-9))    dm_hln = dm * corr    p = 2 * (1 - stats.t.cdf(abs(dm_hln), df=max(n - 1, 1)))    return dict(dm=dm_hln, p=p, dbar=dbar, n=n)def intervalos_empiricos(bt, serie, metodo, quantis=(0.05, 0.25, 0.75, 0.95)):    """Quantis do erro RELATIVO por horizonte, medidos no backtest.    Intervalo teorico de modelo assume residuo bem comportado. O empirico    assume so que o passado se parece com o futuro - premissa mais fraca,    e a unica que sobrevive a um metodo como o gradient boosting.    """    d = bt[(bt.serie == serie) & (bt.metodo == metodo)].copy()    d["rel"] = (d.previsto - d.real) / d.real.clip(lower=1)    out = []    for hh, g in d.groupby("horizonte"):        linha = {"horizonte": hh, "n": len(g), "dp_rel": g.rel.std()}        for q in quantis:            linha[f"q{int(q*100):02d}"] = g.rel.quantile(q)        out.append(linha)    return pd.DataFrame(out)

In [ ]:
import timed = pd.read_csv("vendas_linha.csv", parse_dates=["semana"])ev = pd.read_csv("calendario.csv", parse_dates=["semana"])series = {k: g.sort_values("semana").pecas_liquidas.to_numpy(float)          for k, g in d.groupby("linha")}print(f"series: {len(series)}   semanas: {len(ev)}   "      f"origens por serie: {len(origens(len(ev)))}   horizonte: {H}")t0 = time.time()bt, ordens = rodar(series, ev)print(f"\nbacktest em {time.time() - t0:.0f}s   {len(bt):,} previsoes"      .replace(",", "."))bt.to_csv("backtest.csv", index=False)met = metricas(bt)met.to_csv("metricas.csv", index=False)print("\nMASE por serie e metodo (menor e melhor; 1,0 = igual ao naive sazonal)")piv = met.pivot(index="metodo", columns="serie", values="MASE")piv["media"] = piv.mean(axis=1)piv = piv.sort_values("media")print(piv.round(3).to_string())pd.Series(ordens).to_csv("ordens_sarima.csv")

---## 5. Significância, horizonte e estoqueDiebold-Mariano com correção HAC e de amostra pequena, erro por horizonte, e odimensionamento do estoque de segurança a partir do erro **acumulado** em treze semanas— não da soma dos erros semanais, que superestimaria por supor independência.

In [ ]:
import numpy as npimport pandas as pdfrom scipy import stats# --- parametros de operacao da confeccaoLEAD_TIME = 10        # semanas entre encomendar e receberREVISAO = 3           # a cada quantas semanas se encomenda de novoPROTECAO = LEAD_TIME + REVISAO      # = 13, o horizonte do estudoNIVEL_SERVICO = 0.95CUSTO_PECA = 38.0     # custo unitario medio de producao, em R$def razao_naive(met):    """MASE relativo ao naive sazonal: 0,80 = 20% menos erro que o naive."""    base = met[met.metodo == "Naive sazonal"].set_index("serie").MASE    m = met.copy()    m["vs_naive"] = m.MASE / m.serie.map(base)    return mdef tabela_dm(bt, met, campeao="Regressão harmônica"):    linhas = []    for serie in sorted(bt.serie.unique()):        for m2 in sorted(bt.metodo.unique()):            if m2 == campeao:                continue            r = diebold_mariano(bt, serie, campeao, m2)            linhas.append(dict(serie=serie, contra=m2, dm=r["dm"], p=r["p"],                               vence=r["dbar"] < 0, significativo=r["p"] < 0.05))    return pd.DataFrame(linhas)def erro_acumulado(bt, serie, metodo, protecao=PROTECAO):    """Erro da SOMA das 13 semanas - e isso que dimensiona estoque.    Somar o desvio de cada semana isolada superestima, porque os erros nao    sao independentes: um viés de nivel se repete nas 13 semanas em vez de    se cancelar. Medir a soma diretamente evita a suposicao.    """    d = bt[(bt.serie == serie) & (bt.metodo == metodo)           & (bt.horizonte <= protecao)]    g = d.groupby("origem").agg(real=("real", "sum"), prev=("previsto", "sum"))    g["erro"] = g.prev - g.real    return gdef dimensionar(bt, series, metodo, nivel=NIVEL_SERVICO, protecao=PROTECAO):    z = stats.norm.ppf(nivel)    linhas = []    for serie in sorted(bt.serie.unique()):        g = erro_acumulado(bt, serie, metodo, protecao)        sigma = g.erro.std(ddof=1)        demanda = g.real.mean()        es = z * sigma        linhas.append(dict(            serie=serie, metodo=metodo,            demanda_protecao=demanda,            dp_erro=sigma,            cv_erro=sigma / demanda,            estoque_seguranca=es,            ponto_pedido=demanda + es,            cobertura_semanas=es / (demanda / protecao),            capital_seguranca=es * CUSTO_PECA,        ))    return pd.DataFrame(linhas)

In [ ]:
bt = pd.read_csv("backtest.csv")met = metricas(bt)met = razao_naive(met)met["familia"] = met.metodo.map(FAMILIA)met.to_csv("metricas.csv", index=False)print("=" * 74)print("1. RANKING — MASE medio e razao contra o naive sazonal")print("=" * 74)r = (met.groupby(["metodo", "familia"])     .agg(MASE=("MASE", "mean"), vs_naive=("vs_naive", "mean"),          EPAM=("EPAM", "mean"), vies=("vies", "mean"))     .reset_index().sort_values("MASE"))r["ganho"] = 1 - r.vs_naiver.to_csv("ranking.csv", index=False)print(r.assign(MASE=r.MASE.round(3), vs_naive=r.vs_naive.round(3),               EPAM=(r.EPAM * 100).round(1), vies=(r.vies * 100).round(1),               ganho=(r.ganho * 100).round(1)).to_string(index=False))campeao = r.metodo.iloc[0]print(f"\ncampeao: {campeao}")# ------------------------------------------------------------------ 2. DMprint("\n" + "=" * 74)print("2. DIEBOLD-MARIANO — a vantagem sobrevive ao acaso?")print("=" * 74)dm = tabela_dm(bt, met, campeao)dm.to_csv("diebold_mariano.csv", index=False)piv = dm.pivot(index="contra", columns="serie", values="p")print("valores-p (campeao contra cada metodo, por serie)")print(piv.round(4).to_string())ok = dm.groupby("contra").agg(vitorias=("vence", "sum"),                              significativas=("significativo", "sum"))print("\nem quantas das 5 series o campeao vence, e em quantas com p < 0,05:")print(ok.to_string())# -------------------------------------------------------- 3. por horizonteprint("\n" + "=" * 74)print("3. O ERRO CRESCE COM O HORIZONTE?")print("=" * 74)hz = (bt.assign(rel=lambda d: (d.previsto - d.real).abs() / d.real.clip(lower=1))      .groupby(["metodo", "horizonte"]).rel.mean().reset_index())hz.to_csv("erro_por_horizonte.csv", index=False)p = hz[hz.metodo.isin([campeao, "Naive sazonal", "SARIMA",                       "Holt-Winters multiplicativo"])]print((p.pivot(index="horizonte", columns="metodo", values="rel") * 100)      .round(1).to_string())ints = pd.concat([intervalos_empiricos(bt, s, campeao).assign(serie=s)                  for s in sorted(bt.serie.unique())])ints.to_csv("intervalos.csv", index=False)# ------------------------------------------------------------ 4. estoqueprint("\n" + "=" * 74)print(f"4. DA PREVISAO AO ESTOQUE — lead time {LEAD_TIME} semanas + "      f"revisao {REVISAO}, nivel de servico {NIVEL_SERVICO:.0%}")print("=" * 74)dim_c = dimensionar(bt, None, campeao)dim_n = dimensionar(bt, None, "Naive sazonal")comp = dim_c.merge(dim_n, on="serie", suffixes=("_campeao", "_naive"))comp["economia_pecas"] = (comp.estoque_seguranca_naive                          - comp.estoque_seguranca_campeao)comp["economia_reais"] = comp.economia_pecas * CUSTO_PECAcomp["reducao"] = comp.economia_pecas / comp.estoque_seguranca_naivecomp.to_csv("estoque.csv", index=False)def br(x, c=0):    return f"{x:,.{c}f}".replace(",", "X").replace(".", ",").replace("X", ".")print(f"{'linha':22s} {'demanda 13s':>12s} {'ES campeao':>11s} "      f"{'ES naive':>10s} {'economia':>10s} {'reducao':>8s}")for _, x in comp.iterrows():    print(f"{x.serie:22s} {br(x.demanda_protecao_campeao):>12s} "          f"{br(x.estoque_seguranca_campeao):>11s} "          f"{br(x.estoque_seguranca_naive):>10s} "          f"{br(x.economia_pecas):>10s} {x.reducao:>7.1%}")tot_c = comp.estoque_seguranca_campeao.sum()tot_n = comp.estoque_seguranca_naive.sum()print(f"{'TOTAL':22s} {br(comp.demanda_protecao_campeao.sum()):>12s} "      f"{br(tot_c):>11s} {br(tot_n):>10s} "      f"{br(tot_n - tot_c):>10s} {(tot_n - tot_c) / tot_n:>7.1%}")print(f"\ncapital de giro liberado: R$ "      f"{br((tot_n - tot_c) * CUSTO_PECA, 2)}  "      f"(a R$ {br(CUSTO_PECA, 2)} por peca)")

---## 6. O mesmo protocolo, por canalE aqui o campeão quebra. O Marketplace B é um canal novo, em rampa, com apenas trêsorigens de backtest — e a extrapolação de tendência da regressão harmônica dispara.

In [ ]:
dc = pd.read_csv("vendas_canal.csv", parse_dates=["semana"])series_canal = {k: g.sort_values("semana").pecas_liquidas.to_numpy(float)                for k, g in dc.groupby("canal")}# O Marketplace B so existe a partir da semana 70. Treinar com os zeros iniciais# ensinaria o modelo a prever zero.series_canal["Marketplace B"] = series_canal["Marketplace B"][70:]bt_c, _ = rodar(series_canal, ev)bt_c.to_csv("backtest_canal.csv", index=False)met_c = metricas(bt_c)base_c = met_c[met_c.metodo == "Naive sazonal"].set_index("serie").MASEmet_c["vs_naive"] = met_c.MASE / met_c.serie.map(base_c)met_c.to_csv("metricas_canal.csv", index=False)piv = met_c.pivot(index="metodo", columns="serie", values="vs_naive")piv["media"] = piv.mean(axis=1)print("razao contra o naive sazonal, por canal (menor e melhor):")print(piv.sort_values("media").round(3).to_string())

---## O que este notebook mostra, em uma fraseQue a escolha do método de série temporal é uma **decisão de protocolo**, não de gosto:definido o horizonte pelo lead time, fixado o backtest de origem móvel e escolhida amétrica antes de olhar o resultado, o vencedor aparece sozinho — e, neste caso, elevence porque sabe em que semana cai a Black Friday, não porque é mais sofisticado.E que o mesmo vencedor pode quebrar na série seguinte. "Melhor método" é propriedade dasérie, não do catálogo.